# Library Import

In [2]:
from stable_baselines3 import TD3
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.env_util import make_vec_env
import gymnasium as gym
import os, re
from statistics import mean, stdev
import matplotlib.pyplot as plt

# Walker2D training with TD3

In [2]:
env = make_vec_env("Walker2d-v5", n_envs=8, vec_env_cls=DummyVecEnv)
model = TD3("MlpPolicy", env, 
            learning_rate=1e-3,
            buffer_size=200_000,
            learning_starts=10_000,
            batch_size=256,
            tau=5e-3,
            gamma=0.99,
            train_freq=1,
            gradient_steps=2, 
            action_noise=None, 
            n_steps=1,
            policy_delay=2,
            target_policy_noise=0.2,
            target_noise_clip=0.5,
            verbose=2)
model.learn(total_timesteps=1_000_000)

Using cuda device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.8     |
|    ep_rew_mean     | -2.73    |
| time/              |          |
|    episodes        | 4        |
|    fps             | 10089    |
|    time_elapsed    | 0        |
|    total_timesteps | 192      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 24.8     |
|    ep_rew_mean     | 0.977    |
| time/              |          |
|    episodes        | 8        |
|    fps             | 10003    |
|    time_elapsed    | 0        |
|    total_timesteps | 256      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.9     |
|    ep_rew_mean     | -0.401   |
| time/              |          |
|    episodes        | 12       |
|    fps             | 10051    |
|    time_elapsed    | 0        |
|    total_timesteps | 352    

# Save the Trained Model

In [4]:
BASE_DIR = os.getcwd()
RESULT_FOLDER = 'walker_TD3_SB_results'
RESULT_DIR = os.path.join(BASE_DIR, RESULT_FOLDER)
existing_runs = [d for d in os.listdir(RESULT_DIR) if os.path.exists(os.path.join(RESULT_DIR,d))]
run_numbers = [int(re.search(r'run_(\d{5})',d).group(1)) for d in existing_runs if re.match(r'run_\d{5}',d)]
# model.save('reacher')

trial_number = max(run_numbers, default=-1) + 1
model.save(f'{RESULT_FOLDER}/run_{trial_number:05d}')

# Simulate the Loaded Model

In [ ]:
model_load = TD3.load('walker_TD3_SB_results/run_00000')

width = 1920
height = 1080
default_camera_config = {"azimuth" : 90.0, "elevation" : 0.0, "distance" : 10, "lookat" : [0.0, 0.0, 1.0]}
camera_id = 2

vec_env = gym.make("Walker2d-v5", render_mode='human', 
                    width=width,height=height,
                    default_camera_config=default_camera_config,
                    camera_id=camera_id,

                    forward_reward_weight=1,     # weighting factor of the moving forward reward
                    ctrl_cost_weight=1e-3,        # weighting factor of the large action penalty
                    healthy_reward=1,
                    terminate_when_unhealthy=True,
                    healthy_z_range=(0.8,2),
                    healthy_angle_range=(-1,1),
                    reset_noise_scale=5e-3,       # scale of random pertubations in the initial state
                    exclude_current_positions_from_observation=True,
                    # frame_skip=2, 
                    # camera_name="camera",
                    max_episode_steps=1000
                    )

reward_hist = []
n_test_eps = 1
for eps in range(n_test_eps):
    obs, _ = vec_env.reset()
    dones = False
    rewards = 0

    while not dones:
        action, _ = model_load.predict(obs, deterministic=True)
        nobs, rew, term, trunc, _ = vec_env.step(action)
        done = term or trunc

        obs = nobs if not dones else vec_env.reset()
        rewards += rew
        if done:
            msg = 'done due to termination' if term else 'done due to truncation'
    
    print(f'episode {eps:3d} ' + msg)
    reward_hist.append(rewards)

vec_env.close()

: 

In [10]:
reward_hist

[np.float64(31762.19212581608),
 np.float64(26958.36808941665),
 np.float64(7902.40647285005),
 np.float64(3891.035181007281),
 np.float64(11386.527479183022)]

In [6]:
mean(reward_hist), stdev(reward_hist)

(np.float64(9691.234974926687), 6035.906962476021)

# Analyze the training history

In [ ]:
from stable_baselines3.common.utils import ts2